# VLS Benchmark — Soft Split Analysis & Visualization

Parallel notebook to `analysis_visualization.ipynb` (hard split). Covers:
1. **Soft split classical ML** — RF, GBM, SVM classification + regression
2. **Boltzina** — Uni-Dock/Vina docking + Boltz-2 affinity scoring (10 targets)
3. **GNINA comparison** — CNN rescoring on matched targets (P09211, Q13490)
4. **Predicted vs experimental structure** — Boltz-2 scoring with both receptor types

The soft split (protein_partition) allows related proteins in train and test,
making it an easier benchmark than the hard split.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_auc_score

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FIGSIZE = (10, 5)

# ── paths ──────────────────────────────────────────────────────────────
ROOT = Path("../").resolve()
BOLTZINA_ROOT = Path("/home/aoxu/projects/VLS-Benchmark-Dataset-boltzina")

REGISTRY_SOFT = ROOT / "training_data_full/registry_soft_split_regression.csv"
SOFT_CLASS_DIR = ROOT / "trained_models/soft_split_classification"
SOFT_REG_DIR   = ROOT / "trained_models/regression"
BOLTZINA_DIR   = BOLTZINA_ROOT / "benchmarks/05_boltzina/results/raw_results"
BOLTZINA_POC   = BOLTZINA_ROOT / "benchmarks/05_boltzina/results/poc_proteins.json"
GNINA_DIR      = Path("/tmp/gnina_results")
BOLTZ_CMP_A    = Path("/tmp/boltz_cmpA/results/raw_results")  # predicted
BOLTZ_CMP_B    = Path("/tmp/boltz_cmpB/results/raw_results")  # experimental

COLORS = {
    "random_forest": "#4C72B0", "gradient_boosting": "#DD8452", "svm": "#55A868",
    "boltzina": "#9467bd", "gnina": "#2ca02c", "vina": "#d62728",
}
model_labels = {"random_forest": "RF", "gradient_boosting": "GBM", "svm": "SVM"}

print("ROOT:", ROOT)
print("Soft registry:", REGISTRY_SOFT.exists())
print("Soft classification:", SOFT_CLASS_DIR.exists())
print("Boltzina POC:", BOLTZINA_POC.exists())


## 1  Soft Split Dataset Overview

In [ ]:
reg = pd.read_csv(REGISTRY_SOFT)
print(f"Total rows: {len(reg):,}")
print(f"Unique proteins: {reg['uniprot_id'].nunique()}")
print(f"Unique SMILES: {reg['smiles'].nunique()}")
print()

# Split breakdown
split_summary = reg.groupby(['split', 'is_active']).size().unstack(fill_value=0)
split_summary.columns = ['Decoy', 'Active']
split_summary['Total'] = split_summary.sum(axis=1)
split_summary['Active %'] = (split_summary['Active'] / split_summary['Total'] * 100).round(1)
print(split_summary)

fig, ax = plt.subplots(figsize=(8, 4))
split_summary[['Active', 'Decoy']].plot(kind='bar', stacked=True, ax=ax,
    color=['#2ca02c', '#d62728'], alpha=0.8)
ax.set_title('Soft Split: Active vs Decoy per Split', fontweight='bold')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for i, (_, row) in enumerate(split_summary.iterrows()):
    ax.text(i, row['Total'] + 1000, f"{row['Active %']}% active", ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 2  Classical ML — Soft Split Classification

In [ ]:
# Load soft split classification results
ml_class_rows = []
for model_name in ['random_forest', 'gradient_boosting', 'svm']:
    summary_file = SOFT_CLASS_DIR / f'{model_name}_training_summary.json'
    if not summary_file.exists():
        continue
    with open(summary_file) as f:
        d = json.load(f)
    h = d['training_history']
    row = {'model': model_name}
    for split in ['train', 'val', 'test']:
        for k, v in h.get(f'{split}_metrics', {}).items():
            row[f'{split}_{k}'] = v
    row['n_train'] = h.get('n_train_samples')
    row['n_val'] = h.get('n_val_samples')
    row['n_test'] = h.get('n_test_samples')
    ml_class_rows.append(row)

ml_class = pd.DataFrame(ml_class_rows)
print('Soft split classification results:')
print(ml_class[['model', 'train_roc_auc', 'val_roc_auc', 'test_roc_auc',
                'test_ef_1pct', 'test_bedroc_a80']].round(4).to_string(index=False))

In [ ]:
# ROC-AUC: train / val / test grouped bar
models = ml_class['model'].tolist()
model_labels = {'random_forest': 'RF', 'gradient_boosting': 'GBM', 'svm': 'SVM'}
x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=FIGSIZE)
for i, (split, color) in enumerate([('train', '#4C72B0'), ('val', '#DD8452'), ('test', '#55A868')]):
    vals = ml_class[f'{split}_roc_auc'].values
    bars = ax.bar(x + i * width, vals, width, label=split.capitalize(), color=color, alpha=0.85)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.3f}', ha='center', fontsize=8)

ax.axhline(0.5, color='black', ls=':', lw=1, label='Random')
ax.set_ylabel('ROC-AUC')
ax.set_title('Soft Split Classification — ROC-AUC by Split', fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels([model_labels.get(m, m) for m in models])
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Enrichment factor and BEDROC
ef_cols = [c for c in ml_class.columns if c.startswith('test_ef_')]
bedroc_cols = [c for c in ml_class.columns if c.startswith('test_bedroc_')]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# EF at multiple cutoffs
ax = axes[0]
for _, row in ml_class.iterrows():
    m = row['model']
    cutoffs = [float(c.replace('test_ef_', '').replace('pct', '')) for c in ef_cols]
    vals = [row[c] for c in ef_cols]
    ax.plot(cutoffs, vals, 'o-', label=model_labels.get(m, m), color=COLORS.get(m, 'grey'), lw=2)
ax.axhline(1.0, color='black', ls=':', lw=1)
ax.set_xlabel('Top % cutoff')
ax.set_ylabel('Enrichment Factor')
ax.set_title('Soft Split — Enrichment Factor Curve', fontweight='bold')
ax.legend()

# BEDROC
ax = axes[1]
bedroc_labels = {c: c.replace('test_bedroc_', 'a=').replace('a', 'a=') for c in bedroc_cols}
x = np.arange(len(bedroc_cols))
for i, (_, row) in enumerate(ml_class.iterrows()):
    m = row['model']
    vals = [row[c] for c in bedroc_cols]
    ax.bar(x + i*0.25, vals, 0.25, label=model_labels.get(m, m), color=COLORS.get(m, 'grey'), alpha=0.85)
ax.set_xticks(x + 0.25)
ax.set_xticklabels([c.replace('test_bedroc_', '') for c in bedroc_cols])
ax.set_ylabel('BEDROC')
ax.set_title('Soft Split — BEDROC', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

## 3  Classical ML — Soft Split Regression

In [ ]:
# Load regression results
ml_reg_rows = []
for model_name in ['random_forest', 'gradient_boosting', 'svm']:
    summary_file = SOFT_REG_DIR / f'{model_name}_regressor_training_summary.json'
    if not summary_file.exists():
        continue
    with open(summary_file) as f:
        d = json.load(f)
    h = d['training_history']
    row = {'model': model_name}
    for split in ['train', 'val', 'test']:
        for k, v in h.get(f'{split}_metrics', {}).items():
            row[f'{split}_{k}'] = v
    ml_reg_rows.append(row)

ml_reg = pd.DataFrame(ml_reg_rows)
print('Soft split regression results:')
print(ml_reg[['model', 'test_r2', 'test_rmse', 'test_mae',
              'test_pearson', 'test_spearman', 'test_ci']].round(4).to_string(index=False))

In [ ]:
# Regression metrics bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (col, label) in zip(axes, [
    ('test_r2', 'Test R²'),
    ('test_spearman', 'Test Spearman r'),
    ('test_rmse', 'Test RMSE (lower=better)'),
]):
    vals = ml_reg[col].values
    colors = [COLORS.get(m, 'grey') for m in ml_reg['model']]
    bars = ax.bar([model_labels.get(m, m) for m in ml_reg['model']], vals, color=colors, alpha=0.85)
    ax.set_title(label, fontweight='bold')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
    if col == 'test_r2':
        ax.axhline(0, color='black', ls=':', lw=1)

fig.suptitle('Soft Split Regression — pChEMBL Prediction', fontweight='bold')
plt.tight_layout()
plt.show()

## 4  Boltzina (Boltz-2 Affinity) — Per-Target Results

In [ ]:
# Load boltzina per-protein results
def compute_all_metrics(csv_path):
    """Compute ROC-AUC, EF@1%, EF@5% for all score columns."""
    df = pd.read_csv(csv_path)
    df['is_active'] = df['ligand_name'].str.contains('/actives/').astype(int)
    labels = df['is_active'].values
    n_act = labels.sum()
    n = len(labels)
    if n_act == 0 or n_act == n:
        return None
    
    results = {'n_actives': n_act, 'n_decoys': n - n_act}
    for col, direction, short in [
        ('affinity_probability_binary', 'higher', 'prob_binary'),
        ('affinity_pred_value', 'higher', 'pred_value'),
        ('docking_score', 'lower', 'docking_score'),
    ]:
        scores = -df[col].values if direction == 'lower' else df[col].values
        auc = roc_auc_score(labels, scores)
        n1 = max(1, int(np.ceil(n * 0.01)))
        top1 = np.argsort(scores)[::-1][:n1]
        ef1 = (labels[top1].sum() / n1) / (n_act / n)
        n5 = max(1, int(np.ceil(n * 0.05)))
        top5 = np.argsort(scores)[::-1][:n5]
        ef5 = (labels[top5].sum() / n5) / (n_act / n)
        results[f'{short}_auc'] = auc
        results[f'{short}_ef1'] = ef1
        results[f'{short}_ef5'] = ef5
    return results

boltz_rows = []
with open(BOLTZINA_POC) as f:
    poc_proteins = json.load(f)

for p in poc_proteins:
    uid = p['uniprot_id']
    csv = BOLTZINA_DIR / uid / 'boltzina_results.csv'
    if not csv.exists():
        csv = BOLTZ_CMP_A / uid / 'boltzina_results.csv'
    if not csv.exists():
        continue
    m = compute_all_metrics(csv)
    if m:
        m['uniprot_id'] = uid
        boltz_rows.append(m)

boltz_df = pd.DataFrame(boltz_rows)
print(f'Boltzina results: {len(boltz_df)} targets')
print(boltz_df[['uniprot_id', 'n_actives', 'n_decoys',
                'prob_binary_auc', 'pred_value_auc', 'docking_score_auc',
                'prob_binary_ef1']].round(3).to_string(index=False))

In [ ]:
# Per-target ROC-AUC heatmap for Boltzina scores
score_cols = ['prob_binary_auc', 'pred_value_auc', 'docking_score_auc']
score_labels = ['Boltz-2\nprob_binary', 'Boltz-2\npred_value', 'Vina\ndocking_score']

hm_data = boltz_df.set_index('uniprot_id')[score_cols].T
hm_data.index = score_labels

fig, ax = plt.subplots(figsize=(max(10, len(boltz_df)*1.2), 3))
sns.heatmap(hm_data.astype(float), annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Boltzina Per-Target ROC-AUC (Soft Split Targets)', fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Aggregate Boltzina metrics
print('Boltzina aggregate (mean across targets):')
for col, label in [
    ('prob_binary_auc', 'prob_binary ROC-AUC'),
    ('pred_value_auc', 'pred_value ROC-AUC'),
    ('docking_score_auc', 'docking_score ROC-AUC'),
    ('prob_binary_ef1', 'prob_binary EF@1%'),
    ('docking_score_ef1', 'docking_score EF@1%'),
]:
    print(f'  {label:30s} {boltz_df[col].mean():.3f} +/- {boltz_df[col].std():.3f}')

## 5  Boltzina vs GNINA vs ML — Head-to-Head (P09211, Q13490)

In [ ]:
# Head-to-head on the 2 targets with all three methods
COMPARE_TARGETS = ['P09211', 'Q13490']

compare_rows = []
for uid in COMPARE_TARGETS:
    # Boltzina
    csv = BOLTZ_CMP_A / uid / 'boltzina_results.csv'
    if csv.exists():
        df = pd.read_csv(csv)
        df['is_active'] = df['ligand_name'].str.contains('/actives/').astype(int)
        labels = df['is_active'].values
        for col, direction, name in [
            ('affinity_probability_binary', 'higher', 'Boltzina prob_binary'),
            ('affinity_pred_value', 'higher', 'Boltzina pred_value'),
            ('docking_score', 'lower', 'Boltzina Vina score'),
        ]:
            scores = -df[col].values if direction == 'lower' else df[col].values
            auc = roc_auc_score(labels, scores)
            n = len(scores); n_act = labels.sum()
            n1 = max(1, int(np.ceil(n*0.01))); top1 = np.argsort(scores)[::-1][:n1]
            ef1 = (labels[top1].sum()/n1)/(n_act/n)
            compare_rows.append({'target': uid, 'method': name, 'roc_auc': auc, 'ef_1pct': ef1, 'type': 'boltzina'})

    # GNINA
    csv = GNINA_DIR / uid / 'gnina_results.csv'
    if csv.exists():
        df = pd.read_csv(csv)
        df['is_active'] = df['ligand_name'].str.contains('/actives/').astype(int)
        labels = df['is_active'].values
        for col, direction, name in [
            ('cnn_score', 'higher', 'GNINA cnn_score'),
            ('cnn_affinity', 'higher', 'GNINA cnn_affinity'),
            ('vina_score', 'lower', 'GNINA Vina score'),
        ]:
            scores = -df[col].values if direction == 'lower' else df[col].values
            auc = roc_auc_score(labels, scores)
            n = len(scores); n_act = labels.sum()
            n1 = max(1, int(np.ceil(n*0.01))); top1 = np.argsort(scores)[::-1][:n1]
            ef1 = (labels[top1].sum()/n1)/(n_act/n)
            compare_rows.append({'target': uid, 'method': name, 'roc_auc': auc, 'ef_1pct': ef1, 'type': 'gnina'})

compare_df = pd.DataFrame(compare_rows)
print(compare_df.pivot_table(index='method', columns='target', values='roc_auc').round(3).to_string())

In [ ]:
# Grouped bar: ROC-AUC per method per target
TYPE_COLORS = {
    'boltzina': '#9467bd', 'gnina': '#2ca02c',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, label in zip(axes, ['roc_auc', 'ef_1pct'], ['ROC-AUC', 'EF @ 1%']):
    for i, uid in enumerate(COMPARE_TARGETS):
        sub = compare_df[compare_df['target'] == uid].copy()
        sub = sub.sort_values('roc_auc', ascending=False)
        colors = [TYPE_COLORS.get(t, 'grey') for t in sub['type']]
        y = np.arange(len(sub))
        ax_sub = axes[i] if metric == 'roc_auc' else axes[i]
    
    # Use first target for left plot, second for right
    for ax_idx, uid in enumerate(COMPARE_TARGETS):
        ax = axes[ax_idx]
        sub = compare_df[compare_df['target'] == uid].sort_values(metric, ascending=True)
        colors = ['#9467bd' if 'Boltzina' in m else '#2ca02c' for m in sub['method']]
        y = np.arange(len(sub))
        ax.barh(y, sub[metric], color=colors, alpha=0.85)
        ax.set_yticks(y)
        ax.set_yticklabels(sub['method'], fontsize=9)
        ax.set_title(f'{uid} — {label}', fontweight='bold')
        if metric == 'roc_auc':
            ax.axvline(0.5, color='black', ls=':', lw=1)
        elif metric == 'ef_1pct':
            ax.axvline(1.0, color='black', ls=':', lw=1)
        for yi, v in zip(y, sub[metric]):
            ax.text(v + 0.01, yi, f'{v:.2f}', va='center', fontsize=8)
    break  # only ROC-AUC

fig.suptitle('Boltzina vs GNINA — ROC-AUC per Target', fontweight='bold')
plt.tight_layout()
plt.show()

## 6  Predicted vs Experimental Structure (Boltzina)

In [ ]:
# Compare Boltz-2 scores with predicted vs experimental receptor structure
struct_rows = []
for uid in COMPARE_TARGETS:
    for cond_dir, cond_label in [(BOLTZ_CMP_A, 'Predicted'), (BOLTZ_CMP_B, 'Experimental')]:
        csv = cond_dir / uid / 'boltzina_results.csv'
        if not csv.exists():
            continue
        df = pd.read_csv(csv)
        df['is_active'] = df['ligand_name'].str.contains('/actives/').astype(int)
        labels = df['is_active'].values
        for col, direction, score_label in [
            ('affinity_probability_binary', 'higher', 'prob_binary'),
            ('affinity_pred_value', 'higher', 'pred_value'),
            ('docking_score', 'lower', 'docking_score'),
        ]:
            scores = -df[col].values if direction == 'lower' else df[col].values
            auc = roc_auc_score(labels, scores)
            struct_rows.append({
                'target': uid, 'structure': cond_label, 'score': score_label, 'roc_auc': auc,
            })

struct_df = pd.DataFrame(struct_rows)
pivot = struct_df.pivot_table(index=['target', 'score'], columns='structure', values='roc_auc')

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax, color=['#9467bd', '#ff7f0e'], alpha=0.85)
ax.set_title('Boltz-2 ROC-AUC: Predicted vs Experimental Structure', fontweight='bold')
ax.axhline(0.5, color='black', ls=':', lw=1)
ax.set_xlabel('')
ax.set_xticklabels([f'{t}\n{s}' for t, s in pivot.index], rotation=30, ha='right', fontsize=9)
ax.legend(title='Receptor structure')
plt.tight_layout()
plt.show()

print(pivot.round(3).to_string())

## 7  Score Distributions — Actives vs Decoys

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for row, uid in enumerate(COMPARE_TARGETS):
    # Boltzina
    boltz_csv = BOLTZ_CMP_A / uid / 'boltzina_results.csv'
    bdf = pd.read_csv(boltz_csv)
    bdf['is_active'] = bdf['ligand_name'].str.contains('/actives/')

    # GNINA
    gdf = pd.read_csv(GNINA_DIR / uid / 'gnina_results.csv')
    gdf['is_active'] = gdf['ligand_name'].str.contains('/actives/')

    for col_idx, (df, col, label, color) in enumerate([
        (bdf, 'affinity_probability_binary', f'{uid} — Boltz-2 prob_binary', '#9467bd'),
        (gdf, 'cnn_score', f'{uid} — GNINA cnn_score', '#2ca02c'),
        (bdf, 'docking_score', f'{uid} — Vina docking_score', '#4C72B0'),
    ]):
        ax = axes[row, col_idx]
        act = df.loc[df['is_active'], col].dropna()
        dec = df.loc[~df['is_active'], col].dropna()
        ax.hist(dec, bins=30, alpha=0.6, label=f'Decoys (n={len(dec)})', color='grey', density=True)
        ax.hist(act, bins=30, alpha=0.7, label=f'Actives (n={len(act)})', color=color, density=True)
        ax.set_title(label, fontweight='bold', fontsize=10)
        ax.legend(fontsize=8)
        if col_idx == 0:
            ax.set_ylabel('Density')

fig.suptitle('Score Distributions: Actives vs Decoys', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## 8  Summary — All Methods on Soft Split

In [ ]:
# Unified summary table
summary = []

# ML classification (global metrics)
for _, row in ml_class.iterrows():
    summary.append({
        'method': f"ML {model_labels.get(row['model'], row['model'])}",
        'type': 'ML (classification)',
        'test_roc_auc': row.get('test_roc_auc'),
        'test_ef_1pct': row.get('test_ef_1pct'),
        'test_bedroc_a80': row.get('test_bedroc_a80'),
    })

# ML regression
for _, row in ml_reg.iterrows():
    summary.append({
        'method': f"ML {model_labels.get(row['model'], row['model'])} (reg)",
        'type': 'ML (regression)',
        'test_r2': row.get('test_r2'),
        'test_spearman': row.get('test_spearman'),
    })

# Boltzina aggregate
for score, col in [('prob_binary', 'prob_binary_auc'), ('pred_value', 'pred_value_auc'),
                   ('docking_score', 'docking_score_auc')]:
    summary.append({
        'method': f'Boltzina {score}',
        'type': 'Docking + neural',
        'test_roc_auc': boltz_df[col].mean(),
        'test_ef_1pct': boltz_df[f'{score}_ef1'].mean() if f'{score}_ef1' in boltz_df else None,
    })

summary_df = pd.DataFrame(summary)
print('=== Soft Split — All Methods Summary ===')
print(summary_df.to_string(index=False))